# D2C Skincare Subscription Analytics â€” Data Cleaning & Audit

## Objective

This notebook will:
- Audit the intentionally messy raw data
- Clean and standardize the data
- Validate data quality
- Export cleaned data for analysis

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# Define data paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / 'data/raw'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data/processed'

# Create processed directory if it doesn't exist
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'Raw data directory: {RAW_DATA_DIR.resolve()}')
print(f'Processed data directory: {PROCESSED_DATA_DIR.resolve()}')

In [ ]:
# Load raw CSV files
users = pd.read_csv(RAW_DATA_DIR / 'users.csv')
subscription_events = pd.read_csv(RAW_DATA_DIR / 'subscription_events.csv')
orders = pd.read_csv(RAW_DATA_DIR / 'orders.csv')
marketing_spend = pd.read_csv(RAW_DATA_DIR / 'marketing_spend.csv')

print('All raw datasets loaded successfully.')

In [ ]:
# Display basic information for each dataset
print('=== Raw Data Summary ===')
print(f'\nUsers:')
print(f'  Rows: {len(users):,}')
print(f'  Columns: {list(users.columns)}')

print(f'\nSubscription Events:')
print(f'  Rows: {len(subscription_events):,}')
print(f'  Columns: {list(subscription_events.columns)}')

print(f'\nOrders:')
print(f'  Rows: {len(orders):,}')
print(f'  Columns: {list(orders.columns)}')

print(f'\nMarketing Spend:')
print(f'  Rows: {len(marketing_spend):,}')
print(f'  Columns: {list(marketing_spend.columns)}')

## Raw Data Audit

Read-only audit of the four raw DataFrames (`users`, `subscription_events`, `orders`, `marketing_spend`).
**No cleaning happens in this section** - no duplicates are dropped, no nulls are filled, no channels or dates are standardized.

Checks performed for every table:
1. Row count
2. Column names
3. Data types
4. Null count and null % by column (plus total nulls)
5. Duplicate row count

Table-specific checks:
6. `users` - duplicate `user_id` count and acquisition-channel distribution
7. `subscription_events` - mixed date-format evidence and null `event_type` count
8. `orders` - `user_id` values that do not exist in `users.user_id` and `order_value <= 0`
9. `marketing_spend` - missing month/channel combinations vs the expected 36 months x 4 canonical channels

The section ends with a concise before-cleaning audit summary table.

In [ ]:
# ------------------------------------------------------------------
# Items 1-5: generic per-table audit (READ-ONLY).
# Nothing is modified: no duplicates dropped, no nulls filled.
# ------------------------------------------------------------------

def audit_dataframe(df, name):
    """Print a structural audit of a raw DataFrame and return summary stats."""
    row_count = len(df)
    col_names = list(df.columns)
    null_counts = df.isnull().sum()
    total_nulls = int(null_counts.sum())
    total_cells = row_count * len(col_names)
    duplicate_rows = int(df.duplicated().sum())

    print(f'=== {name} ===')
    print(f'1) Row count: {row_count:,}')
    print(f'2) Column names: {col_names}')
    print('3) Data types:')
    print(df.dtypes.to_string())

    null_pct = (null_counts / row_count * 100).round(2) if row_count > 0 else null_counts.astype(float)
    print('4) Null count / null % by column:')
    print(pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct}).to_string())
    overall_pct = (total_nulls / total_cells * 100) if total_cells else 0.0
    print(f'   Total nulls: {total_nulls:,} ({overall_pct:.2f}% of all cells)')

    print(f'5) Duplicate rows: {duplicate_rows:,}')

    return {
        'rows': row_count,
        'n_columns': len(col_names),
        'total_nulls': total_nulls,
        'null_pct_of_cells': round(overall_pct, 2),
        'duplicate_rows': duplicate_rows,
    }

RAW_TABLES = {
    'users': users,
    'subscription_events': subscription_events,
    'orders': orders,
    'marketing_spend': marketing_spend,
}

table_audits = {name: audit_dataframe(df, name) for name, df in RAW_TABLES.items()}


In [ ]:
# ------------------------------------------------------------------
# Table-specific integrity checks (READ-ONLY).
# ------------------------------------------------------------------

# 6) users - duplicate user_id count and acquisition-channel distribution
users_dup_id_rows = int(users['user_id'].duplicated().sum())  # extra copies beyond the first occurrence
users_dup_id_distinct = int(users.loc[users['user_id'].duplicated(), 'user_id'].nunique())
print('6) users - duplicate user_id and acquisition-channel checks')
print(f'   Rows with an already-seen user_id: {users_dup_id_rows:,}')
print(f'   Distinct user_ids affected:         {users_dup_id_distinct:,}')
print('   Acquisition-channel distribution:')
print(users['acquisition_channel'].value_counts(dropna=False).to_string())

# 7) subscription_events - mixed date-format evidence and null event_type count
subscription_event_dates = subscription_events['event_date'].astype(str)
subscription_date_format_counts = pd.Series(
    np.select(
        [
            subscription_event_dates.str.match(r'^\d{4}-\d{2}-\d{2}'),
            subscription_event_dates.str.match(r'^\d{2}/\d{2}/\d{4}'),
        ],
        ['ISO-like YYYY-MM-DD', 'slash-like DD/MM/YYYY'],
        default='other or unparseable',
    )
).value_counts()
subscription_null_event_types = int(subscription_events['event_type'].isna().sum())
print('\n7) subscription_events - date-format and null event_type checks')
print('   Date-format evidence:')
print(subscription_date_format_counts.to_string())
print(f'   Null event_type rows: {subscription_null_event_types:,}')

# 8) orders - orphan user_ids and non-positive order values
orphan_mask = ~orders['user_id'].isin(users['user_id'])
orphan_rows = int(orphan_mask.sum())
orphan_id_count = int(orders.loc[orphan_mask, 'user_id'].nunique())
print('\n8) orders - orphan user_id check (referential integrity)')
print(f'   Order rows whose user_id is NOT in users: {orphan_rows:,}')
print(f'   Distinct unknown user_ids referenced:     {orphan_id_count:,}')

nonpositive_mask = orders['order_value'] <= 0
nonpositive_rows = int(nonpositive_mask.sum())
print('\n   orders - order_value <= 0 check')
print(f'   Rows with order_value <= 0: {nonpositive_rows:,}')
if nonpositive_rows > 0:
    print(orders.loc[nonpositive_mask, ['order_id', 'user_id', 'order_value']].head(5).to_string(index=False))

# 9) marketing_spend - missing month/channel combinations vs the expected grid.
EXPECTED_MONTHS = pd.period_range('2025-01', '2027-12', freq='M')
EXPECTED_CHANNELS = [
    'Referral', 'Instagram Ads', 'Google Ads', 'Organic',
]
expected_pairs = {(m, c) for m in EXPECTED_MONTHS for c in EXPECTED_CHANNELS}

parsed_months = pd.to_datetime(marketing_spend['month'], errors='coerce')
unparseable_months = int(parsed_months.isna().sum())
actual_pairs = set(zip(parsed_months.dt.to_period('M'), marketing_spend['acquisition_channel']))
missing_pairs = sorted(expected_pairs - actual_pairs, key=lambda p: (str(p[0]), str(p[1])))
unexpected_pairs = sorted(actual_pairs - expected_pairs, key=lambda p: (str(p[0]), str(p[1])))
dup_combo_rows = int(marketing_spend.duplicated(subset=['month', 'acquisition_channel']).sum())

print('\n9) marketing_spend - month/channel completeness check')
print(f'   Expected grid: {len(EXPECTED_MONTHS)} months x {len(EXPECTED_CHANNELS)} channels = {len(expected_pairs):,} combinations')
print(f'   Actual rows: {len(marketing_spend):,} covering {len(actual_pairs):,} distinct combinations')
print(f'   Missing month/channel combinations: {len(missing_pairs):,}')
if missing_pairs:
    print(pd.DataFrame(missing_pairs, columns=['month', 'acquisition_channel']).to_string(index=False))
print(f'   Unexpected (out-of-grid) combinations: {len(unexpected_pairs):,}')
print(f'   Duplicate month/channel rows within the table: {dup_combo_rows:,}')
print(f'   Unparseable month values: {unparseable_months:,}')

In [ ]:
# ------------------------------------------------------------------
# Concise audit summary - main before-cleaning counts (READ-ONLY).
# 'NaN' means the check is not applicable to that table.
# ------------------------------------------------------------------

table_order = ['users', 'subscription_events', 'orders', 'marketing_spend']
audit_summary = pd.DataFrame(
    {
        'rows':            [table_audits[t]['rows'] for t in table_order],
        'columns':         [table_audits[t]['n_columns'] for t in table_order],
        'total_nulls':     [table_audits[t]['total_nulls'] for t in table_order],
        'null_pct_cells':  [table_audits[t]['null_pct_of_cells'] for t in table_order],
        'duplicate_rows':  [table_audits[t]['duplicate_rows'] for t in table_order],
        'dup_user_id_rows':             [users_dup_id_rows, np.nan, np.nan, np.nan],
        'orphan_user_id_rows':          [np.nan, np.nan, orphan_rows, np.nan],
        'order_value_le_0_rows':        [np.nan, np.nan, nonpositive_rows, np.nan],
        'missing_month_channel_combos': [np.nan, np.nan, np.nan, len(missing_pairs)],
    },
    index=table_order,
)
print('=== Raw Data Audit Summary (before cleaning) ===')
audit_summary
